# Titanic Survival Prediction

### Project Overview

The Titanic disaster is one of the most well-known shipwrecks in history.

This project predicts whether a passenger survived the Titanic disaster using various Machine Learning algorithms.

---

## Objectives

- Explore the dataset
- Clean missing values
- Perform feature engineering
- Train multiple ML models
- Compare their performances
- Select the best model
- Generate Kaggle submission

---

## Algorithms Used

- Logistic Regression
- SGD Classifier
- Decision Tree
- Random Forest

---

## Evaluation Metrics

- Accuracy
- Precision
- Recall
- F1 Score
- ROC AUC
- Confusion Matrix

In [70]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve

import os

In [71]:
!pip install -q wandb

import wandb
from kaggle_secrets import UserSecretsClient

WANDB_PROJECT = "titanic-survival-prediction"

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_key)

print("W&B login successful!")

ConnectionError: Connection error trying to communicate with service.

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

In [ ]:
train.head()

In [ ]:
train.sample(10)

In [ ]:
train.shape

In [ ]:
test.shape

In [ ]:
train.info()

In [ ]:
train.describe().T

In [ ]:
train.isnull().sum()

**EDA**

In [ ]:
plt.figure(figsize=(6,5))

sns.countplot(
    data=train,
    x="Survived",
    palette="Set2"
)

plt.title("Survival Count")
plt.show()

In [ ]:
plt.figure(figsize=(6,5))

sns.countplot(
    data=train,
    x="Sex",
    hue="Survived",
    palette="viridis"
)

plt.title("Gender vs Survival")
plt.show()

In [ ]:
plt.figure(figsize=(6,5))

sns.countplot(
    data=train,
    x="Pclass",
    hue="Survived",
    palette="Set1"
)

plt.title("Passenger Class vs Survival")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    train["Age"],
    bins=30,
    kde=True,
    color="royalblue"
)

plt.title("Age Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    train["Fare"],
    bins=40,
    kde=True,
    color="green"
)

plt.title("Fare Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

sns.boxplot(
    data=train,
    x="Survived",
    y="Age",
    palette="coolwarm"
)

plt.title("Age vs Survival")
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

sns.boxplot(
    data=train,
    x="Pclass",
    y="Fare",
    palette="Set3"
)

plt.title("Fare by Passenger Class")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.heatmap(
    train.isnull(),
    cbar=False,
    cmap="viridis"
)

plt.title("Missing Values")
plt.show()

In [ ]:
plt.figure(figsize=(10,7))

corr = train.select_dtypes(include=np.number).corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
train.groupby("Sex")["Survived"].mean().sort_values(ascending=False)

# Data Cleaning & Feature Engineering

In this section I did:

- Handle missing values
- Create new informative features
- Encode categorical variables
- Prepare the dataset for Machine Learning

In [ ]:
test_ids = test["PassengerId"]

In [ ]:
full_data = pd.concat([train, test], axis=0, ignore_index=True)

print(full_data.shape)

In [ ]:
full_data.isnull().sum().sort_values(ascending=False)

In [ ]:
full_data["Age"] = full_data["Age"].fillna(
    full_data["Age"].median()
)

In [ ]:
full_data["Embarked"] = full_data["Embarked"].fillna(
    full_data["Embarked"].mode()[0]
)

In [ ]:
full_data["Fare"] = full_data["Fare"].fillna(
    full_data["Fare"].median()
)

In [ ]:
full_data["Cabin"] = full_data["Cabin"].fillna("Unknown")

In [ ]:
full_data["FamilySize"] = (
    full_data["SibSp"] +
    full_data["Parch"] +
    1
)

In [ ]:
full_data["IsAlone"] = (
    full_data["FamilySize"] == 1
).astype(int)

In [ ]:
full_data["Title"] = full_data["Name"].str.extract(
    " ([A-Za-z]+)\.",
    expand=False
)

In [ ]:
full_data["Title"].value_counts()

In [ ]:
full_data["Title"] = full_data["Title"].replace(
[
'Lady','Countess','Capt','Col',
'Don','Dr','Major','Rev',
'Sir','Jonkheer','Dona'
],
'Rare'
)

full_data["Title"] = full_data["Title"].replace(
'Ms','Miss'
)

full_data["Title"] = full_data["Title"].replace(
'Mlle','Miss'
)

full_data["Title"] = full_data["Title"].replace(
'Mme','Mrs'
)

In [ ]:
full_data["Sex"] = LabelEncoder().fit_transform(
    full_data["Sex"]
)

In [ ]:
full_data["Embarked"] = LabelEncoder().fit_transform(
    full_data["Embarked"]
)

In [ ]:
full_data["Title"] = LabelEncoder().fit_transform(
    full_data["Title"]
)

In [ ]:
full_data["CabinKnown"] = (
    full_data["Cabin"] != "Unknown"
).astype(int)

In [ ]:
full_data.drop(
[
"PassengerId",
"Name",
"Ticket",
"Cabin"
],
axis=1,
inplace=True
)

In [ ]:
train = full_data.iloc[:891].copy()

test = full_data.iloc[891:].copy()

In [ ]:
X = train.drop("Survived", axis=1)

y = train["Survived"]

test = test.drop("Survived", axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)

print("Training Labels   :", y_train.shape)
print("Testing Labels    :", y_test.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

test_scaled = scaler.transform(test)

In [ ]:
pd.DataFrame(
    X_train,
    columns=X.columns
).head()

# Model Training

I will train multiple machine learning algorithms and compare their performance.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SGD Classifier": SGDClassifier(loss="log_loss", random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )
}

In [ ]:
results = []
trained_models = {}

for name, model in models.items():

    print("=" * 60)
    print(f"Training: {name}")
    print("=" * 60)

    # SGD uses scaled features
    if name == "SGD Classifier":
        X_tr = X_train_scaled
        X_te = X_test_scaled
    else:
        X_tr = X_train
        X_te = X_test

    
    run = wandb.init(
        project=WANDB_PROJECT,
        name=name.lower().replace(" ", "_"),
        job_type="train",
        config={
            "model": name,
            "random_state": 42,
            "test_size": 0.20,
            "scaled_features": name == "SGD Classifier"
        },
        reinit=True
    )

  
    model.fit(X_tr, y_train)

  
    pred = model.predict(X_te)

    acc = accuracy_score(y_test, pred)

    report = classification_report(
        y_test,
        pred,
        output_dict=True
    )

    precision = report["weighted avg"]["precision"]
    recall = report["weighted avg"]["recall"]
    f1 = report["weighted avg"]["f1-score"]

   
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_te)[:, 1]
    else:
        y_score = model.decision_function(X_te)

    roc_auc = roc_auc_score(y_test, y_score)

  
    wandb.log({
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "roc_auc": roc_auc
    })

   
    cm = confusion_matrix(y_test, pred)

    wandb.log({
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_test,
            preds=pred,
            class_names=["Not Survived", "Survived"]
        )
    })

    
    results.append([
        name,
        acc,
        precision,
        recall,
        f1,
        roc_auc
    ])

    trained_models[name] = model

    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"ROC AUC   : {roc_auc:.4f}")

 
    run.finish()

In [ ]:
results = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC AUC"
    ]
)

results.sort_values(
    by="Accuracy",
    ascending=False,
    inplace=True
)

results

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=results,
    x="Accuracy",
    y="Model",
    palette="viridis"
)

plt.title("Titanic Model Accuracy Comparison")
plt.xlabel("Accuracy")
plt.ylabel("Model")
plt.xlim(0.70, 1.0)

for i, value in enumerate(results["Accuracy"]):
    plt.text(
        value + 0.005,
        i,
        f"{value:.4f}",
        va="center"
    )

plt.show()

In [ ]:
comparison_run = wandb.init(
    project=WANDB_PROJECT,
    name="model_comparison",
    job_type="eval",
    reinit=True
)


comparison_table = wandb.Table(
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC AUC"
    ],
    data=results.values.tolist()
)


comparison_run.log({
    "model_comparison": comparison_table
})

accuracy_chart = wandb.plot.bar(
    comparison_table,
    "Model",
    "Accuracy",
    title="Titanic Model Accuracy Comparison"
)

comparison_run.log({
    "accuracy_comparison": accuracy_chart
})

comparison_run.finish()

In [ ]:
best_model_name = results.iloc[0]["Model"]

print("Best Model :", best_model_name)

In [ ]:
best_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

best_model.fit(X_train_scaled, y_train)

y_pred = best_model.predict(X_test_scaled)

print("Best Model: Logistic Regression")
print("Model trained successfully!")

In [ ]:
if best_model_name == "SGD Classifier":
    y_pred = best_model.predict(X_test_scaled)
else:
    y_pred = best_model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Logistic Regression Accuracy: {accuracy:.4f}")

In [ ]:
cv_scores = cross_val_score(
    LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    X_train_scaled,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Cross Validation Scores:")
print(cv_scores)

print(f"\nMean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Survived", "Survived"]
))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=1,
    linecolor="black",
    xticklabels=["Not Survived", "Survived"],
    yticklabels=["Not Survived", "Survived"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Logistic Regression")

plt.show()

In [ ]:
y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

print("Prediction probabilities generated successfully.")

In [ ]:
roc_auc = roc_auc_score(y_test, y_prob)

print(f"ROC AUC Score: {roc_auc:.4f}")

In [ ]:
fpr, tpr, thresholds = roc_curve(
    y_test,
    y_prob
)

plt.figure(figsize=(7, 6))

plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {roc_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    "--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")

plt.legend()
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_prob
)

plt.figure(figsize=(7, 6))

plt.plot(
    recall,
    precision
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Logistic Regression")

plt.show()

In [ ]:
coefficients = pd.Series(
    best_model.coef_[0],
    index=X.columns
).sort_values()

coefficients

In [ ]:
plt.figure(figsize=(9, 6))

coefficients.plot.barh()

plt.axvline(
    0,
    linestyle="--"
)

plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.title("Feature Impact - Logistic Regression")

plt.show()



In [ ]:
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"]
}

grid = GridSearchCV(
    LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train)

print("Best Parameters:")
print(grid.best_params_)

print(f"\nBest CV Accuracy: {grid.best_score_:.4f}")

In [ ]:
final_model = LogisticRegression(
    **grid.best_params_,
    max_iter=2000,
    random_state=42
)

final_model.fit(
    X_train_scaled,
    y_train
)

print("Final Logistic Regression model trained.")

In [ ]:
final_pred = final_model.predict(X_test_scaled)

final_accuracy = accuracy_score(
    y_test,
    final_pred
)

print(f"Final Validation Accuracy: {final_accuracy:.4f}")

In [ ]:
X_scaled_full = scaler.fit_transform(X)
test_scaled_final = scaler.transform(test)
final_model.fit(
    X_scaled_full,
    y
)

print("Final model trained on complete training dataset.")

In [ ]:
test_predictions = final_model.predict(
    test_scaled_final
)

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": test_predictions
})

submission.head(10)

In [ ]:
submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully!")

In [ ]:
print("FINAL MODEL RESULTS")
print("Model               : Logistic Regression")
print(f"Validation Accuracy : {final_accuracy:.4f}")
print(f"Cross Validation    : {grid.best_score_:.4f}")
print(f"Best Parameters     : {grid.best_params_}")


# Final Conclusion

## Best Model: Logistic Regression

After comparing multiple machine learning algorithms, Logistic Regression achieved the best validation performance on the Titanic dataset.

### Key Results

- **Model:** Logistic Regression
- **Validation Accuracy:** 0.8156
- **Cross-Validation Accuracy:** 0.7964
- **ROC AUC:** 0.8581
- **Hyperparameters:** Optimized using GridSearchCV

### Key Insights

- Gender was an important predictor of survival.
- Passenger class had a strong relationship with survival.
- Family-related features provided additional predictive information.
- Feature engineering improved the model inputs.
- Logistic Regression provides interpretable coefficients that help explain feature influence.

### Final Workflow

```text
Data Collection
      ↓
Exploratory Data Analysis
      ↓
Data Cleaning
      ↓
Feature Engineering
      ↓
Encoding
      ↓
Feature Scaling
      ↓
Model Comparison
      ↓
Logistic Regression
      ↓
Cross Validation
      ↓
Hyperparameter Tuning
      ↓
Final Model
      ↓
Kaggle Prediction
      ↓
submission.csv